In [3]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.feature_extraction import FeatureHasher
from sklearn.preprocessing import PolynomialFeatures

sns.set_style("whitegrid")

In [4]:
data = pd.read_csv('Train.csv')

print('Dataset Size :', data.shape)
display(data.head())

Dataset Size : (10999, 12)


,ID,Warehouse_block,Mode_of_Shipment,Customer_care_calls,Customer_rating,Cost_of_the_Product,Prior_purchases,Product_importance,Gender,Discount_offered,Weight_in_gms,Reached.on.Time_Y.N
0,1,D,Flight,4,2,177,3,low,F,44,1233,1
1,2,F,Flight,4,5,216,2,low,M,59,3088,1
2,3,A,Flight,2,2,183,4,low,M,48,3374,1
3,4,B,Flight,3,3,176,4,medium,M,10,1177,1
4,5,C,Flight,2,2,184,3,medium,F,46,2484,1


In [5]:
print('Columns in Dataset:\n', data.columns)

print('\nMissing Values:\n')
print(data.isnull().sum())

Columns in Dataset:
 Index(['ID', 'Warehouse_block', 'Mode_of_Shipment', 'Customer_care_calls',
       'Customer_rating', 'Cost_of_the_Product', 'Prior_purchases',
       'Product_importance', 'Gender', 'Discount_offered', 'Weight_in_gms',
       'Reached.on.Time_Y.N'],
      dtype='object')

Missing Values:

ID                     0
Warehouse_block        0
Mode_of_Shipment       0
Customer_care_calls    0
Customer_rating        0
Cost_of_the_Product    0
Prior_purchases        0
Product_importance     0
Gender                 0
Discount_offered       0
Weight_in_gms          0
Reached.on.Time_Y.N    0
dtype: int64


In [6]:
shipment_values = data['Mode_of_Shipment'].astype(str)

tokens = []
for value in shipment_values:
    tokens.append([value])

In [7]:
feature_hasher = FeatureHasher(n_features=10, input_type='string')

hashed_result = feature_hasher.transform(tokens)

hashed_array = hashed_result.toarray()

In [8]:
hash_columns = ['Shipment_hash_' + str(i) for i in range(10)]

hashed_df = pd.DataFrame(hashed_array, columns=hash_columns)

print('Hashed Feature Shape :', hashed_df.shape)
display(hashed_df.head())

Hashed Feature Shape : (10999, 10)


,Shipment_hash_0,Shipment_hash_1,Shipment_hash_2,Shipment_hash_3,Shipment_hash_4,Shipment_hash_5,Shipment_hash_6,Shipment_hash_7,Shipment_hash_8,Shipment_hash_9
0,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [9]:
categorical_cols = ['Mode_of_Shipment', 'Warehouse_block']

combined_text = data[categorical_cols].astype(str)

combined_tokens = []
for row in combined_text.values:
    combined_tokens.append(list(row))

In [10]:
multi_hasher = FeatureHasher(n_features=6, input_type='string')

multi_hash = multi_hasher.transform(combined_tokens)

multi_hash_array = multi_hash.toarray()

In [11]:
multi_hash_cols = ['MultiHash_' + str(i) for i in range(6)]

multi_hash_df = pd.DataFrame(multi_hash_array, columns=multi_hash_cols)

print('Multi Hash Shape :', multi_hash_df.shape)
display(multi_hash_df.head())

Multi Hash Shape : (10999, 6)


,MultiHash_0,MultiHash_1,MultiHash_2,MultiHash_3,MultiHash_4,MultiHash_5
0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,-2.0,0.0,0.0,0.0
2,0.0,0.0,-1.0,0.0,1.0,0.0
3,0.0,0.0,-2.0,0.0,0.0,0.0
4,0.0,0.0,-1.0,-1.0,0.0,0.0


In [12]:
data['Cost_per_Gram'] = data['Cost_of_the_Product'] / (data['Weight_in_gms'] + 1)

display(data[['Cost_of_the_Product','Weight_in_gms','Cost_per_Gram']].head())

,Cost_of_the_Product,Weight_in_gms,Cost_per_Gram
0,177,1233,0.143436
1,216,3088,0.069926
2,183,3374,0.054222
3,176,1177,0.149406
4,184,2484,0.074044


In [13]:
data['Discount_Ratio'] = data['Discount_offered'] / (data['Cost_of_the_Product'] + 1)

display(data[['Discount_offered','Cost_of_the_Product','Discount_Ratio']].head())

,Discount_offered,Cost_of_the_Product,Discount_Ratio
0,44,177,0.247191
1,59,216,0.271889
2,48,183,0.260870
3,10,176,0.056497
4,46,184,0.248649


In [14]:
data['Customer_Engagement'] = data['Customer_care_calls'] * data['Customer_rating']

display(data[['Customer_care_calls','Customer_rating','Customer_Engagement']].head())

,Customer_care_calls,Customer_rating,Customer_Engagement
0,4,2,8
1,4,5,20
2,2,2,4
3,3,3,9
4,2,2,4


In [15]:
poly_generator = PolynomialFeatures(degree=2, interaction_only=True)

numeric_cols = ['Cost_of_the_Product','Discount_offered']

poly_matrix = poly_generator.fit_transform(data[numeric_cols])

poly_names = poly_generator.get_feature_names_out(numeric_cols)

poly_df = pd.DataFrame(poly_matrix, columns=poly_names)

display(poly_df.head())

,1,Cost_of_the_Product,Discount_offered,Cost_of_the_Product Discount_offered
0,1.0,177.0,44.0,7788.0
1,1.0,216.0,59.0,12744.0
2,1.0,183.0,48.0,8784.0
3,1.0,176.0,10.0,1760.0
4,1.0,184.0,46.0,8464.0


In [16]:
final_dataset = pd.concat([data, hashed_df, multi_hash_df, poly_df], axis=1)

print('Final Dataset Shape :', final_dataset.shape)

display(final_dataset.head())

Final Dataset Shape : (10999, 35)


,ID,Warehouse_block,Mode_of_Shipment,Customer_care_calls,Customer_rating,Cost_of_the_Product,Prior_purchases,Product_importance,Gender,Discount_offered,...,MultiHash_0,MultiHash_1,MultiHash_2,MultiHash_3,MultiHash_4,MultiHash_5,1,Cost_of_the_Product,Discount_offered,Cost_of_the_Product Discount_offered
0,1,D,Flight,4,2,177,3,low,F,44,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,177.0,44.0,7788.0
1,2,F,Flight,4,5,216,2,low,M,59,...,0.0,0.0,-2.0,0.0,0.0,0.0,1.0,216.0,59.0,12744.0
2,3,A,Flight,2,2,183,4,low,M,48,...,0.0,0.0,-1.0,0.0,1.0,0.0,1.0,183.0,48.0,8784.0
3,4,B,Flight,3,3,176,4,medium,M,10,...,0.0,0.0,-2.0,0.0,0.0,0.0,1.0,176.0,10.0,1760.0
4,5,C,Flight,2,2,184,3,medium,F,46,...,0.0,0.0,-1.0,-1.0,0.0,0.0,1.0,184.0,46.0,8464.0
